# Table of Contents

#### Imported Packages

In [2]:
import numpy as np
import pandas as pd
from surprise import Dataset, Reader, SVD
# Assuming these are your local utility files
from evaluation import rmse_function 
from preprocessing import train_test
from config import RATING_PARQUET_PATH

In [5]:
def train_svd(train_df):
    # 1. Define the Reader - ensure the scale matches your actual data
    reader = Reader(rating_scale=(1, 5))
    
    # 2. Load the training data
    # Surprise expects exactly [user, item, rating] columns in this order
    data = Dataset.load_from_df(train_df[["user_id", "movie_id", "rating"]], reader)
    trainset = data.build_full_trainset()
    

    # 3. Initialize and train SVD
    model = SVD(n_factors=50, reg_all=0.05, n_epochs=1, verbose=True)
    model.fit(trainset)

    return model

In [ ]:
def predict_svd(model, test_df):
    # 4. Efficient Prediction
    # Convert test_df to the list of tuples format Surprise expects: [(uid, iid, r_ui), ...]
    # This is much faster than manual zipping for large dataframes
    testset_tuples = list(test_df[["user_id", "movie_id", "rating"]].itertuples(index=False, name=None))
    predictions = model.test(testset_tuples)

    # 5. Extract estimated ratings
    # .est is the predicted rating
    preds = np.array([pred.est for pred in predictions])
    return preds


In [ ]:
from evaluation import rmse_function
import pandas as pd

svd = pd.read_parquet("../predictions/residual_svd_preds.parquet")
knn = pd.read_parquet("../predictions/knn_preds.parquet")
bias = pd.read_parquet("../predictions/bias_model_preds.parquet")

df = svd.merge(knn, on=["user_id", "movie_id"], suffixes=("_svd", "_knn"))
df = df.merge(bias, on=["user_id", "movie_id"])
df = df.rename(columns={"pred": "pred_bias"})

w_1=0.6
w_2=0.3
w_3=0.1

df["ensemble"] = (
    w_1 * df["pred_svd"]
    + w_2 * df["pred_knn"]
    + w_3 * df["pred_bias"]
)

rmse = rmse_function(df["actual_svd"], df["ensemble"])
print(f"Ensemble RMSE: {rmse:.4f}")

Ensemble RMSE: 0.9361


In [16]:
# We use integers 1-10 to represent 0.1-1.0 to avoid float precision issues
min = 1

for w1_int in range(0, 11):
    for w2_int in range(0, 11):
        for w3_int in range(0, 11):
            if w1_int + w2_int + w3_int == 10:
                # Convert back to decimals for printing
                w1, w2, w3 = w1_int / 10, w2_int / 10, w3_int / 10
                
                df["ensemble"] = (
                w1 * df["pred_svd"]
                + w2 * df["pred_knn"]
                + w3 * df["pred_bias"]
                )

                rmse = rmse_function(df["actual_svd"], df["ensemble"])
                print(f"Ensemble RMSE: {rmse:.4f}")

                if rmse < min:
                    min = rmse
                    best_weights = (w1, w2, w3)

print(f"Best weights: {best_weights}")
print(f"Best RMSE: {min:.4f}")

Ensemble RMSE: 0.9656
Ensemble RMSE: 0.9600
Ensemble RMSE: 0.9552
Ensemble RMSE: 0.9512
Ensemble RMSE: 0.9480
Ensemble RMSE: 0.9457
Ensemble RMSE: 0.9441
Ensemble RMSE: 0.9435
Ensemble RMSE: 0.9436
Ensemble RMSE: 0.9446
Ensemble RMSE: 0.9464
Ensemble RMSE: 0.9592
Ensemble RMSE: 0.9541
Ensemble RMSE: 0.9498
Ensemble RMSE: 0.9463
Ensemble RMSE: 0.9437
Ensemble RMSE: 0.9419
Ensemble RMSE: 0.9409
Ensemble RMSE: 0.9408
Ensemble RMSE: 0.9415
Ensemble RMSE: 0.9431
Ensemble RMSE: 0.9535
Ensemble RMSE: 0.9490
Ensemble RMSE: 0.9452
Ensemble RMSE: 0.9423
Ensemble RMSE: 0.9402
Ensemble RMSE: 0.9390
Ensemble RMSE: 0.9386
Ensemble RMSE: 0.9390
Ensemble RMSE: 0.9403
Ensemble RMSE: 0.9487
Ensemble RMSE: 0.9447
Ensemble RMSE: 0.9415
Ensemble RMSE: 0.9391
Ensemble RMSE: 0.9376
Ensemble RMSE: 0.9369
Ensemble RMSE: 0.9371
Ensemble RMSE: 0.9381
Ensemble RMSE: 0.9448
Ensemble RMSE: 0.9413
Ensemble RMSE: 0.9387
Ensemble RMSE: 0.9368
Ensemble RMSE: 0.9359
Ensemble RMSE: 0.9357
Ensemble RMSE: 0.9365
Ensemble R